# 밀도 차이 검정

- 대응 코드: `significance.py`
- 문서별 밀도 분포와 한일·국가 내부 검정을 구분해 확인.
- p값과 FDR 결과를 원문 맥락과 함께 해석.


In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
from IPython.display import display

# 작업 폴더가 저장소 루트이거나 노트북 폴더여도 analysis/를 찾는다.
candidates = [Path.cwd(), *Path.cwd().parents]
analysis_root = next((candidate for base in candidates
                      for candidate in (base, base / "code" / "analysis", base / "ESG_comp" / "code" / "analysis")
                      if (candidate / "common" / "config.py").is_file()), None)
if analysis_root is None:
    raise FileNotFoundError("ESG_comp/code/analysis 폴더에서 노트북을 열어 주세요.")
if not os.environ.get("ESG_RESEARCH_ROOT"):
    raise RuntimeError("비공개 입력 위치를 ESG_RESEARCH_ROOT 환경변수로 먼저 지정하세요.")
sys.path.insert(0, str(analysis_root))
from common import config as cfg
from common.io_common import load_enriched


def show_csv(path, rows=10, columns=None):
    path = Path(path)
    if not path.is_file():
        print(f"저장 결과 없음: {path}\n입력 자료와 앞 단계를 확인한 뒤 실행하세요.")
        return None
    frame = pd.read_csv(path)
    display(frame[columns].head(rows) if columns else frame.head(rows))
    print(f"{path.name}: {len(frame):,}행")
    return frame


In [ ]:
RUN_BETWEEN = False  # 한일 비교 검정
RUN_WITHIN = False   # 국가 내부 검정
PREVIEW_ROWS = 15

In [ ]:
from stage2_frequency import significance as step
from common.io_common import doc_densities
for lang in cfg.LANGS:
    merged, docs = load_enriched(lang)
    main_docs = docs[docs["is_main"]]
    main_mentions = merged[merged["is_main"]]
    density = pd.Series(doc_densities(main_docs, main_mentions) * 1000)
    print(f"{lang}: main 문서 {len(main_docs):,}건, 언급 {len(main_mentions):,}건 (1,000자당)")
    display(density.describe().to_frame("density_per_1000_chars"))

In [ ]:
out_dir = cfg.OUTPUT_DIR / "significance"
if RUN_BETWEEN:
    between = step.build_significance_report(out_dir)
    display(between.head(PREVIEW_ROWS))
if RUN_WITHIN:
    within = step.build_within_country_significance(out_dir)
    display(within.head(PREVIEW_ROWS))
if not (RUN_BETWEEN or RUN_WITHIN):
    print("검정 재실행은 RUN_BETWEEN/RUN_WITHIN에서 선택하세요.")

In [ ]:
show_csv(out_dir / "category_composition_descriptive.csv", PREVIEW_ROWS)
show_csv(out_dir / "significance_tests_all.csv", PREVIEW_ROWS)
show_csv(out_dir / "significance_within_country.csv", PREVIEW_ROWS)